# Point Cloud Classification

[![image](https://colab.research.google.com/assets/colab-badge.svg)](https://githubtocolab.com/opengeos/geoai/blob/main/docs/examples/point_cloud_classification.ipynb)

This notebook demonstrates how to classify 3D point clouds into semantic
categories (e.g., ground, buildings, vegetation) using RandLA-Net via the
`geoai.pointcloud` module.

**Reference:**
Hu et al., "RandLA-Net: Efficient Semantic Segmentation of Large-Scale
Point Clouds," CVPR 2020.

## Install packages

In [ ]:
# %pip install "geoai-py[pointcloud]" leafmap[lidar]

## Import libraries

In [ ]:
import os

import leafmap
import numpy as np

from geoai.pointcloud import (
    ASPRS_CLASSES,
    PointCloudClassifier,
    classify_point_cloud,
    list_pointcloud_models,
)

## List available models

GeoAI ships pre-trained RandLA-Net checkpoints for three common scenarios.

In [ ]:
models = list_pointcloud_models()
for name, desc in models.items():
    print(f"{name}:\n  {desc}\n")

## ASPRS classification codes

The standard LAS classification codes used in airborne LiDAR.

In [ ]:
for code, name in sorted(ASPRS_CLASSES.items()):
    print(f"  {code:3d}: {name}")

## Download sample data

Download a sample LiDAR point cloud for classification.

In [ ]:
url = "https://opengeos.org/data/lidar/madison.zip"
filename = "madison.las"
leafmap.download_file(url, "madison.zip", unzip=True)

## Visualize the input point cloud

Use leafmap to view the raw point cloud before classification.

In [ ]:
leafmap.view_lidar(filename, cmap="terrain", backend="pyvista")

## Initialize the classifier

Create a `PointCloudClassifier` with the Toronto3D pre-trained model,
which is best suited for urban outdoor point clouds.

In [ ]:
classifier = PointCloudClassifier(
    model_name="RandLANet_Toronto3D",
    device="cpu",  # Use "cuda" if you have a GPU
)

## Run classification

Classify the point cloud and save the result to a new LAS file.

In [ ]:
output_path = "madison_classified.las"
labels, probabilities = classifier.classify(
    filename,
    output_path=output_path,
)
print(f"Classified {len(labels)} points into {len(np.unique(labels))} classes.")

## Examine classification results

View summary statistics of the classified point cloud.

In [ ]:
stats = classifier.summary(output_path)
print(f"Total points: {stats['total_points']:,}")
print(f"\nClass distribution:")
for name, pct in stats["class_percentages"].items():
    count = stats["class_counts"][name]
    print(f"  {name:30s}: {count:>8,} ({pct:.1f}%)")

## Visualize classified point cloud

View the classified point cloud colored by class labels.

In [ ]:
classifier.visualize(output_path, backend="pyvista", cmap="tab20")

## Quick classification (convenience function)

For one-off classification, use the `classify_point_cloud()` convenience
function which creates the model and runs inference in a single call.

In [ ]:
# labels, probs = classify_point_cloud(
#     "input.las",
#     output_path="output_classified.las",
#     model_name="RandLANet_Toronto3D",
# )

## Fine-tuning on custom data

If you have labeled LAS/LAZ files, you can fine-tune the model on your
own data.  Place training files in one directory and (optionally)
validation files in another.  Each file must have the `classification`
field populated with ground-truth labels.

In [ ]:
# history = classifier.train(
#     train_dir="path/to/train_data/",
#     val_dir="path/to/val_data/",
#     epochs=50,
#     learning_rate=0.001,
# )
# print("Final training loss:", history["train_loss"][-1])